Пакеты, которые используются в этом блокноте:

# Глава 3: Программирование механизмов внимания

In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.11.0


<img src="https://camo.githubusercontent.com/47358e1fe19859b3a0c9e82bec664f8cbb5650da9b632a7a70fb4c2b6d63ca2c/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30312e776562703f313233" width="800px">

<img src="https://camo.githubusercontent.com/12846bf6a7a8cc9f40e16baf0d91d194c8fd3568dc2601f2371a4d514b81898e/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30322e77656270" width="800px">

## 3.1 Проблема моделирования длинных последовательностей

- Перевод текста дословно невозможен из-за различий в грамматических структурах между исходным и целевым языками:

<img src="https://camo.githubusercontent.com/38734035d68e30c6d6f904352feae79b4f60859bda22d838fc29d5b0a6016703/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30332e77656270" width="800px">

- До появления трансформеров рекурентные нейронные сети (recurrent neural networks, RNN) были самой популярной архитектурой кодировщик-декодировщик для языкового перевода
- При такой настройке кодировщик обрабатывает последовательность токенов из исходного языка, используя скрытое состояние — своего рода промежуточный уровень в нейронной сети — для генерации сжатого представления всей входной последовательности:

<img src="https://camo.githubusercontent.com/65735d9325818c300bec445e93a922c66e3a64760f7ad4a0fedb3cde7991f039/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30342e77656270" width="800px">


**Скрытое состояние** — это **память** или **блокнот** нейронной сети прямо во время чтения предложения.

Вы читаете первое слово «Кот». Ваш блокнот: «Пока речь о коте».

Вы читаете второе слово «сидит». Блокнот обновляется: «Кто-то (кот) совершает действие (сидит)».

Вы читаете третье слово «на». Блокнот: «Кот сидит где-то (пока не знаем где)».

Вы читаете четвертое слово «столе». Блокнот финальный: «Кот сидит на столе».

**Это финальное содержимое блокнота как раз и есть «скрытое состояние» после обработки всего предложения.** Оно хранит в себе *сжатый смысл* всей фразы.

### Зачем это нужно?

Когда сеть-декодировщик (переводчик) начнет рождать английские слова «The cat sits on the...», она не будет перечитывать исходное предложение заново. Вместо этого она просто **посмотрит в ваш блокнот (скрытое состояние)** и поймет: «Ага, субъект — кот, действие — сидеть, место — стол».

### Как это реализовано технически (немного глубже)

Технически «скрытое состояние» — это **просто список чисел** (вектор), например, из 256, 512 или 1024 чисел.

С каждым новым словом происходит три шага:

1.  **Берем старый блокнот** (предыдущее скрытое состояние).
2.  **Берем новое слово** (превращенное в числа — вектор).
3.  **Нейронная сеть (ячейка RNN) выполняет очень простую формулу**:
    `Новый блокнот = функция(Старый блокнот, Новое слово)`

Эта функция — всегда одно и то же уравнение с весами (настройками, которые сеть выучила на миллионах примеров). Она решает: «Как сильно новое слово должно *изменить* старую память?»

**Важная деталь:** Числа внутри «блокнота» — это **не** конкретные слова («кот», «стол»). Это абстрактные признаки, которые сеть придумала сама:
- Первое число может означать «насколько действие уже завершено».
- Второе — «одушевленность субъекта».
- Третье — «активность глагола».
- ...и так далее. Человек эти числа не интерпретирует.

### Проблема RNN

Представьте, что вы читаете длинное предложение:
> «Тот большой рыжий кот, который сломал вчера вазу и которого мы выгнали, ... **сидел** на столе».

К тому моменту, как вы дочитали до слова «сидел», ваш **блокнот уже переполнен** информацией о «сломал», «вазу», «выгнали». А слово «кот» было в самом начале. Старое скрытое состояние уже много раз перезаписалось новыми словами.

Это называется **проблема забывания дальних связей**. RNN очень легко помнит последние 5-7 слов, но «кот» и «сидел» могут «разорваться» в памяти сети. Из-за этого трансформеры (с их вниманием) и пришли на смену RNN — они умеют смотреть прямо на любое слово из прошлого, не полагаясь на «один блокнот на все».

**Скрытое состояние** — это постоянно обновляющаяся «записка памяти», чтобы к концу предложения нейросеть помнила главное о начале.

## 3.2 Управление зависимостей данных с помощью механизмов привлечения внимания

- С помощью механизма внимания декодирующая часть сети, генерирующая текст, может выборочно обращаться ко всем входным токенам. Это означает, что некоторые входные токены более важны для генерации конкретного выходного токена, чем другие. Важность определяется весами внимания:

<img src="https://camo.githubusercontent.com/87f17c8ca381ec022193c5b6acd2c9a7d3cd8b000c632c848e92221c19a4ef00/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30352e77656270" width="800px">

- Самовнимание - механизм, который используется для вычисления более эффективных входных представлений. Он позволяет каждой позиции во входной последовательности взаимодействовать со всеми остальными позициями в той же последовательности и оценивать их вклад (важность) при вычислении представления последовательности

<img src="https://camo.githubusercontent.com/8c1e96b812fd6f79afb05d90a83b57e8c22ae6c8d98ba38420e933ebd20606fa/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30362e77656270" width="800px">

## 3.3 Обращение к разным частям входных данных с помощью самовнимания

### 3.3.1 Простой механизм самовнимания без обучаемых весов

- Предположим, что нам дана входная последовательность от $x ^ {(1)}$ до $x ^ {(T)}$
    - Входные данные представляют собой текст (например, предложение типа "Your journey starts with one step"), который уже преобразован во встроенные токены.
    - Например, $x ^{(1)}$ - это d-мерный вектор, представляющий слово "Ваш", и так далее
- ** Цель: ** вычислить контекстные векторы $z ^{(i)}$ для каждого элемента входной последовательности $x^{(i)}$ от $x ^{(1)}$ до $x^{(T)}$ (где $z$ и $x$ имеют тот же размер)
    - Вектор контекста $z^{(i)}$ представляет собой взвешенную сумму входных данных от $x^{(1)}$ до $x^{(T)}$
    - Вектор контекста является "контекстно" зависимым от определенных входных данных
    - Вместо $x^{(i)}$ в качестве заполнителя для произвольного входного токена давайте рассмотрим второй входной токен, $x^{(2)}$
    - И чтобы продолжить с конкретным примером, вместо заполнителя $z^{(i)}$ мы рассмотрим второй выходной вектор контекста, $z^{(2)}$
    - Второй контекстный вектор, $z ^{(2)}$, представляет собой взвешенную сумму по всем входным данным от $x ^{(1)}$ до $x^{(T)}$, взвешенную по отношению ко второму входному элементу $x^{(2)}$
    - Веса внимания - это веса, которые определяют, какой вклад вносит каждый из входных элементов во взвешенную сумму при вычислении $z ^ {(2)}.$
    - Короче говоря, представьте себе $z ^ {(2)}$ как модифицированную версию $x ^ {(2)}$, которая также включает в себя информацию обо всех других элементах ввода, имеющих отношение к данной задаче

<img src="https://camo.githubusercontent.com/a5f9c3d06c5fdc5345b311a9b0baf0e6601211a32b27822427d3275c4d4ec2d0/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30372e77656270" width="800px">

(Цифры на этом рисунке усечены до одной цифры после запятой, чтобы уменьшить визуальный беспорядок)

- По общему правилу ненормированные значения коэффициента внимания называются **"показателями внимания"**, тогда как нормализованные значения показателя внимания, сумма которых равна 1, называются **"весами внимания".**

- Приведенный ниже код шаг за шагом повторяет приведенный выше рисунок

</br>

- **Шаг 1:** вычислите ненормализованные показатели внимания $\omega$
- Предположим, используется второй входной токен в качестве запроса, то есть $q^{(2)} = x^{(2)}$, вычисляются ненормализованные показатели внимания с помощью точечных произведений:
    - $\omega_{21} = x^{(1)} q^{(2)\top}$
    - $\omega_{22} = x^{(2)} q^{(2)\top}$
    - $\omega_{23} = x^{(3)} q^{(2)\top}$
    - ...
    - $\omega_{2T} = x^{(T)} q^{(2)\top}$
- Выше, $\omega$ - это греческая буква "омега", используемая для обозначения ненормализованных показателей внимания, q - элемент матрицы вложений, x - входное значение
    - Индекс "21" в $\omega_{21}$ означает, что элемент входной последовательности 2 использовался в качестве запроса к элементу входной последовательности 1

- Пусть  есть следующее входное предложение, которое уже встроено в трехмерные векторы:

In [2]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

- Правилаа машинного и глубокого обучения - обучающие примеры представлены в виде строк, а значения объектов - в виде столбцов; в случае тензора, показанного выше, каждая строка представляет слово, а каждый столбец - измерение для встраивания

- Основная цель - продемонстрировать, как вектор контекста $z^{(2)}$ вычисляется с использованием второй входной последовательности, $x^{(2)}$, в виде запроса

- На рисунке показан начальный этап этого процесса, который включает в себя вычисление показателей внимания ω между $x ^ {(2)}$
и всеми другими входными элементами с помощью операции точечного умножения

<img src="https://camo.githubusercontent.com/128687a9a383db05f7715d04f5779621f407e76b668e807da298dfbf42ab3a43/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30382e77656270" width="800px">

- Используется элемент входной последовательности 2, $x ^{(2)}$, в качестве примера для вычисления вектора контекста $z^{(2)}$; позже будет обощение для вычисления всех векторов контекста.
- Первым шагом является вычисление ненормализованных показателей внимания путем вычисления точечного произведения между запросом $x^{(2)}$ и всеми другими входными токенами:

In [3]:
query = inputs[1]  # 2-й входной токен - это запрос

attn_scores_2 = torch.empty(inputs.shape[0]) # создает пустой массив из 3-х ячеек (shape - кортеж из библиотеки PyTorch)
for i, x_i in enumerate(inputs):
    # torch.dot() - скалярное произведение
    attn_scores_2[i] = torch.dot(x_i, query) # точечное произведение (транспонировать здесь не нужно, так как это векторы размером 1 дюйм)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


### Почему именно скалярное произведение? Почему нельзя было просто сложить числа? Почему именно перемножить и сложить?

---

### 1. Интуитивный уровень: "Совпадение интересов"

Представь, что векторы `x` (другие слова) и `q` (запрос) — это два списка ответов на вопросы анкеты. Оценки могут быть **положительными** (нравится) или **отрицательными** (не нравится).

**Пример:**
*   **Запрос (Кот):** Хочу того, кто: [Мягкий: +1, Большой: -1, Игривый: +1].
*   **Слово (Диван):** Я такой: [Мягкий: +1, Большой: +1, Сонный: -1].

**Как работают разные операции:**

1.  **Обычная сумма:** Мы тупо складываем цифры. Это как считать общую сумму баллов, игнорируя знаки.
    *   Результат не покажет конфликт. Если один хочет +100, а другой дает -100, их сумма будет 0, что *кажется* похожим на случай, когда оба хотят 0. Это **плохо**.

2.  **Разность (расстояние):** Мы вычитаем векторы ($x - q$). Мы ищем различия. Чем меньше разница, тем ближе. Это работает, но это **мера различия**, а нам нужна мера **совпадения** и **созвучия**. К тому же у разности теряется идея "усиления" при больших совпадениях.

3.  **Скалярное произведение (Умножение + Сложение):**
    *   Если оба числа имеют **одинаковый знак** (оба любят мягкость: $+1 \times +1 = +1$), мы получаем плюс и прибавляем его. Сотрудничество!
    *   Если знаки **противоположны** (кот хочет маленького, а диван большой: $-1 \times +1 = -1$), мы получаем минус и **штрафуем** результат. Конфликт интересов!
    *   Если кому-то **все равно** (ноль), то умножение на ноль обнуляет этот признак. Мы его не учитываем.

**Вывод:** Скалярное произведение — это **единственная простая операция, которая умеет награждать за согласие и наказывать за противоречие**, суммируя эти "плюсики" и "минусики" в одну общую оценку совместимости.

---

### 2. Геометрический уровень: "Угол обзора"

В школьной математике есть формула скалярного произведения:

$$\mathbf{x} \cdot \mathbf{q} = \|\mathbf{x}\| \cdot \|\mathbf{q}\| \cdot \cos(\theta)$$

Давай переведем:
*   $\|\mathbf{x}\|$ — длина вектора "Слова". (Насколько слово "длинное").
*   $\|\mathbf{q}\|$ — длина вектора "Запроса".
*   $\cos(\theta)$ — косинус угла между ними.

**Что делают сумма/разность?**
Они зависят от того, куда направлены оси координат, и от длины векторов. Два длинных вектора, смотрящие в разные стороны, могут дать такую же сумму, как два коротких, смотрящие в одну. Это хаос.

**Что делает скалярное произведение?**
Оно напрямую зависит от **угла**.
*   Если векторы смотрят **в одну сторону** (угол $0^\circ$, $\cos = 1$): произведение **МАКСИМАЛЬНОЕ**. Слова — синонимы.
*   Если векторы **перпендикулярны** (угол $90^\circ$, $\cos = 0$): произведение равно **НУЛЮ**. Слова не связаны.
*   Если векторы смотрят в **противоположные стороны** (угол $180^\circ$, $\cos = -1$): произведение **МАКСИМАЛЬНО ОТРИЦАТЕЛЬНОЕ**. Слова — антонимы.

А механизму внимания как раз и нужно ловить направление смысла: "Кот" и "Мурлыка" должны смотреть в одну сторону, "Кот" и "Бетон" — в перпендикулярные, "Горячий" и "Холодный" — в противоположные.

---

### 3. Математический уровень: "Билинейная форма"

Если посмотреть на реальную формулу Attention с матрицами $W_q$ и $W_k$:

$$\text{Score} = x_i^T W_q^T W_k x_j$$

Это называется **билинейной формой**. Это самый простой способ заставить нейросеть "обучиться" тому, как именно сравнивать два вектора. 

Матрицы $W$ как бы поворачивают пространство так, чтобы в новом пространстве нужные нам слова (например, глагол и его наречие) стали смотреть в одну сторону. А скалярное произведение — это идеальный инструмент, чтобы в этом новом пространстве померить косинус угла.

---

### Резюме

Скалярное произведение взяли за основу, потому что это **операция измерения "созвучия" (сонаправленности) двух наборов чисел, которая учитывает знаки**.

*   **Сумма** — это как измерять богатство человека, складывая его доходы и долги, не замечая, что долги — это минус.
*   **Скалярное произведение** — это как считать реальный баланс: доходы прибавляем, а долги вычитаем, понимая, что в сумме у него может оказаться минус, и он совсем не похож на богача.

---

- Дополнительное примечание: точечное произведение - это, по сути, сокращение для поэлементного умножения двух векторов и суммирования полученных результатов:

In [4]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


- **Шаг 2:** нормализация ненормализованных показателей внимания ("омеги", $\omega$) таким образом, чтобы они в сумме равнялись 1
- Вот простой способ нормализовать ненормализованные показатели внимания и суммировать их до 1 (условное обозначение, полезное для интерпретации и важное для стабильности обучения).:

<img src="https://camo.githubusercontent.com/bcd4dbb967d31efe0abcb5fcf77b533b3452d52dadaffee16bb74c128107b5b7/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f30392e77656270" width="800px">

In [5]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Набор чисел:", attn_scores_2)
print("Веса внимания:", attn_weights_2_tmp)
print("Сумма:", attn_weights_2_tmp.sum())

Набор чисел: tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
Веса внимания: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Сумма: tensor(1.0000)


### Почему делим тензор на сумму его элементов? 

---

### 1. Магия бродкастинга - "магия PyTorch" (как тензор делится на число?)

Когда ты в PyTorch пишешь:
```python
тензор = torch.tensor([7.4, 20.1, 2.7])
сумма = тензор.sum() # -> 30.2 (одно число)
результат = тензор / сумма
```

Python понимает это по-человечески, а не математически-строго:
> *"Ага, у меня есть список из трех чисел и одно число. Пользователь, наверное, хочет поделить **каждый элемент** списка на это число. Я умный, я сделаю это для каждого по очереди!"*

**Python незримо для тебя разворачивает это в цикл:**
```python
# То, что написано:
attn_weights = scores / scores.sum()

# То, что Python делает внутри (приблизительно):
сумма = scores.sum() # 30.2
результат = []
for x in scores:
    результат.append(x / сумма) # Делим каждый элемент отдельно
```

Это называется **поэлементная операция** (element-wise operation). Ты даешь команду одной строкой, а компьютер применяет деление к каждой клеточке тензора.

---

### 2. Почему нам нужно делить каждую ячейку?

Вернемся к смыслу. До деления у нас просто "сила сигнала" (экспоненты):
`[7.4, 20.1, 2.7]`

Они неудобные. Если мы скажем нейросети: *"Возьми 20.1 частей слова №2 и смешай с 7.4 частями слова №1"*, получится винегрет непонятной концентрации.

Мы хотим сказать: *"Возьми **66%** слова №2 и **24%** слова №1"*.

Чтобы получить эти проценты, мы должны **каждую** силу сигнала (каждый элемент) разделить на **общую** силу (сумму).

**Процесс идет по ячейкам:**
1. Ячейка 0: `7.4 / 30.2 = 0.245` (Стало процентом)
2. Ячейка 1: `20.1 / 30.2 = 0.666` (Стало процентом)
3. Ячейка 2: `2.7 / 30.2 = 0.089` (Стало процентом)

Теперь у нас новый тензор `[0.245, 0.666, 0.089]`, который является нормированной "смесью".

---

### 3. Что было бы, если бы мы не делили?

Если бы этого деления не было, Attention не работал бы как вероятностный механизм.

Допустим, мы не поделили и пошли дальше — смешивать векторы `v` (Values).
В коротком предложении сумма внимания могла бы быть 30, в длинном — 500.
Масштаб выхода слоя Attention зависел бы от длины предложения. Это вызвало бы **взрыв градиентов** (числа становились бы то гигантскими, то крошечными при переходе от слоя к слою), и нейросеть бы просто сломалась (перестала обучаться).

Деление на сумму **фиксирует бюджет внимания** — у каждого токена всегда есть ровно 1 единица внимания, которую он может потратить на все остальные слова (включая себя).

У тебя есть три кучки яблок:
- Куча 1: 7 яблок
- Куча 2: 20 яблок
- Куча 3: 3 яблока
Всего 30 яблок.

Команда `тензор / тензор.sum()` говорит:
"А теперь выброси яблоки и оставь вместо них **долю от общего урожая**".
- 7 / 30 = 0.23 (23% урожая)
- 20 / 30 = 0.67 (67% урожая)
- 3 / 30 = 0.10 (10% урожая)

----

- Рекомендуется использовать функцию softmax для нормализации, которая лучше справляется с экстремальными значениями и обладает более желательными свойствами градиента во время обучения.
- Вот базовая реализация функции softmax для масштабирования, которая также нормализует векторные элементы таким образом, что они суммируются до 1:

In [6]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0) # dim=0  # суммируем "вниз" (по вертикали), сворачивая строки в одну 
                                                  # dim=1  # суммируем "вправо" (по горизонтали), сворачивая столбцы в один

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Веса внимания:", attn_weights_2_naive)
print("Сумма:", attn_weights_2_naive.sum())

Веса внимания: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Сумма: tensor(1.)


### Что делает код выше?

Это **Softmax** — улучшенная версия превращения баллов в проценты. Он решает проблемы, которые есть у простого деления на сумму.

---

### Строка 1-2: Функция `softmax_naive`

```python
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
```

**Что здесь происходит по шагам:**

1. **`torch.exp(x)`** — берем **экспоненту** (число $e \approx 2.718$) и возводим её в степень каждого элемента вектора `x`.
   - Для числа 2: $e^2 \approx 7.39$
   - Для числа 0: $e^0 = 1$
   - Для числа -2: $e^{-2} \approx 0.14$

2. **`torch.exp(x).sum(dim=0)`** — считаем сумму всех экспонент.

3. **`torch.exp(x) / сумма`** — делим каждую экспоненту на общую сумму.

Отличный вопрос! Ты спрашиваешь про разницу между **простым делением на сумму** и **Softmax**, а также про выбор `dim=0` или `dim=1`.

Давай разложим по полочкам, **что, когда и зачем применять**.

### `dim=0` vs `dim=1`

Представь тензор как таблицу:

```python
tensor = [
    [a, b, c],   # строка 0
    [d, e, f]    # строка 1
]
# shape = (2 строки, 3 столбца)
```

- **`dim=0`** → иду **вниз по строкам** (сворачиваю вертикально)
- **`dim=1`** → иду **вправо по столбцам** (сворачиваю горизонтально)

### Когда `dim=0`?

Суммируем **каждый столбец отдельно**. Результат — одна строка.

**Применяй, когда:**
- Хочешь узнать сумму/среднее по каждому **признаку** (колонке)
- Нормализуешь **батч** данных (каждый столбец отдельно)
- В Attention: **почти никогда** для весов внимания

**Пример:**
```python
оценки = torch.tensor([
    [5, 4, 3],  # ученик 1: матем, физика, инглиш
    [2, 5, 4]   # ученик 2
])

# dim=0: средний балл по каждому ПРЕДМЕТУ
оценки.sum(dim=0)  # [7, 9, 7] — сумма баллов по предметам
```

---

### Когда `dim=1`?

Суммируем **каждую строку отдельно**. Результат — один столбец.

**Применяй, когда:**
- Хочешь узнать сумму/среднее для каждого **объекта** (строки)
- **В Attention — всегда `dim=1` или `dim=-1`** для весов внимания!
- Нормализуешь вероятности внутри одного запроса

**Пример:**
```python
оценки = torch.tensor([
    [5, 4, 3],  # ученик 1
    [2, 5, 4]   # ученик 2
])

# dim=1: сумма баллов каждого УЧЕНИКА
оценки.sum(dim=1)  # [12, 11] — общий балл каждого
```

В Attention всегда `dim=-1` (последняя ось)

```python
scores = torch.tensor([
    [0.5, 0.2, 0.8],   # запрос 1 → ключи
    [0.1, 0.9, 0.3]    # запрос 2 → ключи
])

# dim=1: каждый запрос получает свои 100% внимания
weights = torch.softmax(scores, dim=1)
# weights[0] = [0.35, 0.26, 0.39]  (сумма = 1)
# weights[1] = [0.25, 0.55, 0.20]  (сумма = 1)
```

`dim=-1` работает как `dim=1` для 2D, но универсальнее: работает для любого числа измерений (всегда берет последнюю ось).


| Ситуация | Инструмент | `dim` |
|----------|------------|-------|
| Учебный пример, всё положительное | `x / sum(x)` | не важно |
| Реальная нейросеть, любые числа | `softmax(x)` | `dim=-1` |
| Один вектор (одномерный) | любой метод | `dim=0` |
| Матрица attention, нормируем запросы | `softmax(x, dim=-1)` | `dim=-1` |
| Нормируем батч или признаки | зависит от задачи | `dim=0` или `dim=1` |

---

### Строка 3: Применяем функцию

```python
attn_weights_2_naive = softmax_naive(attn_scores_2)
```

Берем наши сырые баллы (например, `[0.48, 0.74, 0.35]`) и пропускаем через softmax.

---

### Строка 5-6: Печатаем результат

```python
print("Веса внимания:", attn_weights_2_naive)
print("Сумма:", attn_weights_2_naive.sum())
```

Смотрим на проценты и проверяем, что сумма = 1.

---

### Почему именно `exp(x)`, а не просто `x / sum(x)`?

У простого деления на сумму есть **две проблемы**:

### Проблема 1: Отрицательные числа

Если в баллах есть отрицательное число:

```python
баллы = [5, 2, -3]
сумма = 5 + 2 + (-3) = 4

# Простое деление:
веса = [5/4, 2/4, -3/4] = [1.25, 0.5, -0.75]
```

💥 **Катастрофа!** У нас появился **отрицательный процент** (-0.75), что бессмысленно. Нельзя "уделить -75% внимания" слову.

**Softmax спасает:**
- $e^{-3} \approx 0.05$ (это положительное число, хоть и маленькое!)
- Отрицательные баллы получают мизерные, но **положительные** веса.

---

### Проблема 2: Неразличимость слабых сигналов

Представь, что у нас очень маленькие баллы:

```python
баллы = [0.5, 0.4, 0.1]
сумма = 1.0

# Простое деление:
веса = [0.5, 0.4, 0.1]  # почти не изменилось
```

Все веса очень близки друг к другу. Модель "не уверена", на кого смотреть — всё кажется одинаково важным.

**Softmax с `exp` усиливает контраст:**
- `e^0.5 ≈ 1.65` → вес `1.65 / 4.22 ≈ 0.39`
- `e^0.4 ≈ 1.49` → вес `1.49 / 4.22 ≈ 0.35`
- `e^0.1 ≈ 1.11` → вес `1.11 / 4.22 ≈ 0.26`

Разница стала выразительнее! `exp` "растягивает" пространство вокруг больших чисел.

---

### Проблема 3: Экстремально большие числа

Если приходит очень большой балл:

```python
баллы = [100, 2, 1]
# Простое деление: 100/103 ≈ 0.97, остальные почти 0
```

Всё и так работает, но `exp(100)` — это астрономически огромное число (больше, чем атомов во Вселенной). Компьютер может "захлебнуться".

**На практике** из баллов предварительно вычитают максимум (`x - max(x)`), чтобы избежать слишком больших чисел, но в учебном коде это опускают для простоты.

---

### Сравнение в таблице

| Метод | Формула | Работает с минусами? | Контрастность | Итог |
|-------|---------|---------------------|---------------|------|
| Простое деление | $x / \sum x$ | ❌ Нет | Слабая | Только для учебных примеров |
| **Softmax** | $e^x / \sum e^x$ | ✅ Да | Сильная | **Используется везде** |

---

### Метафора с экзаменом

Представь, что баллы — это оценки за экзамен:

- **Простое деление:** Ученик получил 5, другой 3, третий -2. Делим на общую сумму — третьему достается **отрицательная доля** внимания учителя. Так не бывает!
- **Softmax с exp:** Учитель каждую оценку "усиливает" через экспоненту. Двойка становится крошечным положительным числом (учитель всё равно чуть-чуть посмотрит на двоечника), а пятёрка становится огромной и забирает почти всё внимание. При этом суммарное внимание = 100%.

Обычное деление на сумму — это как раздать всем поровну по куску пиццы, даже тому, кто сказал, что не голоден (и даже тому, кто сказал "я на диете" — ему отрицательный кусок?).

Softmax — это волшебная печка:
1. Она превращает любой ответ в положительное "да" (даже если человек сказал "нет", печка делает это маленьким "да-ам").
2. Она делает громкий голос (большое число) еще громче, а шепот — еще тише.
3. В конце все голоса в сумме дают ровно 1 голос внимания.

---

- Базовая реализация, описанная выше, может страдать от проблем с числовой нестабильностью при больших или малых входных значениях из-за проблем с переполнением и недостаточным расходом
- Следовательно, на практике рекомендуется использовать реализацию PyTorch в softmax, которая была оптимизирована для повышения производительности:

In [7]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Веса внимания:", attn_weights_2)
print("Сумма:", attn_weights_2.sum())

Веса внимания: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Сумма: tensor(1.)


- **Шаг 3**: вычисление вектора контекста $z^{(2)}$ путем умножения встроенных входных токенов, $x^{(i)}$ на значения внимания и суммирование полученных векторов:

<img src="https://camo.githubusercontent.com/477fbbc39086ac657d7f6e6bc176e7f9f27f7784e5e7673c71ca89a0c17c57c4/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f31302e77656270" width="800px">

In [8]:
query = inputs[1] # 2-й входной токен - это запрос

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


### Зачем умножать встроенные входные токены на значения внимания и суммировать полученные вектора? Почему именно такая формула?

---

### 1. Почему нельзя просто "взять слово с самым большим весом" (Hard Attention)?

Допустим, у нас веса: Кот — 0.67, Пушистый — 0.24, Спит — 0.09.

Можно было бы тупо выбрать победителя: **Кот (0.67)** и сказать: «Окей, слово "пушистый" в этом предложении означает "кот"».

**Почему это плохо:**
1.  **Потеря нюансов.** Смысл "пушистого" — это не просто "кот". Это "котовость" (67%) + "пушистость как свойство" (24%) + "сонное состояние" (9%).
2.  **Нет градиента.** Нейросеть учится через плавные изменения. Если мы просто выкидываем 33% информации, мы обрываем связи. Нельзя понять, как чуть-чуть изменить ответ, если мы огрубили результат до одного слова.

---

### 2. Почему нельзя "просто сложить векторы" (как в мешке слов)?

Допустим, мы взяли и тупо смешали все слова в кучу поровну: `x_кот + x_пушистый + x_спит`.

**Почему это плохо:**
Представь, что ты готовишь борщ. "Просто сложить" — это кинуть в кастрюлю все овощи целиком, не чистя, и залить водой.
Результат — "средняя температура по больнице".

*   Слово "Пушистый" теряет свою роль **вопроса**. Оно впитает в себя одинаково и важного "Кота", и неважный предлог "на", и слово "коврик". Получится каша.

**Что делает наша формула:** `sum(w_i * x_i)`.
Мы не просто складываем векторы. Мы **взвешиваем** их.
*   Мы кладем в борщ 67% капусты и только 9% соли. Мы регулируем концентрацию каждого ингредиента, чтобы получился нужный вкус (контекст).

---

### 3. Почему именно Умножение и Сложение? (Геометрия смысла)

Вектор — это точка в пространстве смыслов.
*   Точка "Кот" (вектор `x_0`)
*   Точка "Спит" (вектор `x_2`)

Мы не хотим оказаться ни в точке "Кот", ни в точке "Спит". Мы хотим оказаться **где-то между ними**, но ближе к "Коту".

Формула `w_0 * x_0 + w_2 * x_2` — это математический способ **нарисовать отрезок** между точками "Кот" и "Спит" и поставить новую точку на этом отрезке.

*   Если `w_0 = 1`, а `w_2 = 0` — мы в точке "Кот".
*   Если `w_0 = 0.5`, а `w_2 = 0.5` — мы ровно посередине.
*   Если `w_0 = 0.67` — мы в точке "Кот, который немного спит".

Это называется **выпуклая комбинация**. Она позволяет создать **новый смысл**, которого не было в словаре отдельно, плавно смешав существующие смыслы.

---

### 4. Магия "Остаточного сигнала" (Residual Connection)

В реальном Трансформере эту формулу обычно записывают так:
`Новое_слово = Старое_слово + Смесь_контекста`

То есть к исходному вектору "Пушистый" прибавляют то, что мы насмешивали из контекста.

Твой код `context_vec_2 += attn * x` делает именно это.
Мы не заменили "Пушистый" котом. Мы **обогатили** "Пушистый" знанием о том, что рядом есть кот.

Исходное слово `[0.1, 0.8, 0.3]` (признак "пушистость" горел ярко: 0.8).
После добавления контекста он получил `[0.24, 0.59, 0.48]`.
Он остался "Пушистым" (признаки никуда не делись), но слегка сдвинулся в сторону "Кота" и "Сна".

---

### Итог

Формула `sum(w * x)` гениальна, потому что она решает сразу три задачи:

1.  **Дифференцируемость:** Всё плавно, нейросеть может учиться.
2.  **Интерполяция:** Мы создаем новые смыслы на стыке слов («кот+спящий»), а не просто переключаемся между ними.
3.  **Обогащение:** Мы не теряем исходное слово, а добавляем к нему контекст, делая его умнее.

Если убрать умножение на веса (просто сложить), получится каша. Если убрать сложение всех векторов (оставить только один), мы потеряем нюансы. Только их комбинация дает "понимание".

---